This notebook is to create the TECHGROUPKPOT.inc file for the sensitivity scenario where elctrolizers, electric boilers and heat pumps are restricted.

## 1. Capacity of electrolyzers, electric boilers and heat pumps per country in 2030 and 2035
`G_CAP_YCRAF` (GW) from the unrestricted run. `TECHGROUPKPOT` limits exogenous capacity (minus decommissioning) plus the accumulated new investments, so the table gives both: the **new** capacity (`ENDOGENOUS`, accumulated up to that year, not only built in that year) and the **total installed** capacity (`EXOGENOUS` + `ENDOGENOUS` + `DECOMMISSIONING`), which is the quantity the cap would bind.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
from IPython.display import display
from pybalmorel import MainResults

RUN_FOLDER = "base_dc"                                   # unrestricted run whose investments are used as reference
RESULTS    = "MainResults_fullweek.gdx"
YEARS      = ["2030", "2035"]
RENAME = {"Y": "Year", "C": "Country", "RRR": "Region", "AAA": "Area", "G": "Generation", "FFF": "Fuel",
          "TECH_TYPE": "Technology", "COMMODITY": "Commodity", "UNITS": "Unit"}

res = MainResults(files=RESULTS, paths=str(Path.cwd() / RUN_FOLDER / "model"), scenario_names=[RUN_FOLDER])

def p2x_technology(generation):
    """Technology of interest from the generator name (GNR_ELYS_*, GNR_HP_*, GNR_BO_ELEC_*); None for everything else."""
    g = str(generation).upper()
    if g.startswith("GNR_ELYS"): return "Electrolyzer"
    if g.startswith("GNR_HP_"): return "Heat pump"
    if g.startswith("GNR_BO_ELEC"): return "Electric boiler"
    return None

cap = res.get_result("G_CAP_YCRAF").rename(columns=RENAME)
print("Columns:", list(cap.columns))
# the column holding the variable type (EXOGENOUS / ENDOGENOUS / DECOMMISSIONING) is named differently across pybalmorel versions: find it by its values
var_col = next(c for c in cap.columns if cap[c].isin(["ENDOGENOUS"]).any())
cap = cap.rename(columns={var_col: "Variable"})
cap = cap.assign(Year=cap["Year"].astype(str), Tech=cap["Generation"].map(p2x_technology))
cap = cap.loc[cap["Tech"].notna() & cap["Year"].isin(YEARS)]
print("Variable types:", sorted(cap["Variable"].unique()))   # expected: DECOMMISSIONING, ENDOGENOUS, EXOGENOUS

by_var = cap.groupby(["Country", "Tech", "Year", "Variable"])["Value"].sum().unstack("Variable", fill_value=0)
table = pd.DataFrame({"New (accumulated)": by_var.get("ENDOGENOUS", 0),
                      "Total installed": by_var.sum(axis=1)}).unstack("Year").swaplevel(axis=1).sort_index(axis=1)
table = table.reindex(columns=pd.MultiIndex.from_product([YEARS, ["New (accumulated)", "Total installed"]]), fill_value=0)

# One table per technology (rows = countries, columns = year x quantity), GW
new_capacity = {tech: table.xs(tech, level="Tech") for tech in table.index.get_level_values("Tech").unique()}
for tech, t in new_capacity.items():
    t = t.copy(); t.loc["Total"] = t.sum()
    print(f"{tech}: installed capacity per country (GW)")
    display(t.round(3))


Loading /work3/s243704/Balmorel/base_dc/model/MainResults_fullweek.gdx
Columns: ['Scenario', 'Year', 'Country', 'Region', 'Area', 'Generation', 'Fuel', 'Commodity', 'Technology', 'Category', 'Unit', 'Value']
Variable types: ['DECOMMISSIONING', 'ENDOGENOUS', 'EXOGENOUS']
Electric boiler: installed capacity per country (GW)


2030                              2035  \
                       New (accumulated) Total installed New (accumulated)   
Country                                                                      
ALBANIA                            0.181           0.242             0.181   
AUSTRIA                            1.591           1.869             1.591   
BELGIUM                            0.730           2.078             0.805   
BOSNIA_AND_HERZEGOVINA             0.100           0.116             0.100   
BULGARIA                           1.671           1.779             1.704   
CROATIA                            0.293           0.884             0.417   
CYPRUS                             0.346           0.449             0.346   
CZECH_REPUBLIC                     0.627           0.894             0.679   
DENMARK                            0.671           0.952             0.600   
ESTONIA                            0.057           0.277             0.057   
FINLAND                            6.209           7.205             6.237   
FRANCE                             7.412          18.099             4.048   
GERMANY                           23.786          29.972            19.444   
GREECE                             2.956           4.498             3.073   
HUNGARY                            1.256           1.388             1.349   
IRELAND                            2.257           2.545             1.421   
ITALY                              8.468          18.809             8.468   
LATVIA                             0.000           0.118             0.000   
LITHUANIA                          0.418           0.546             0.197   
LUXEMBOURG                         0.501           0.553             0.477   
MALTA                              0.054           0.086             0.054   
MONTENEGRO                         0.049           0.069             0.049   
NETHERLANDS                        2.250           3.124             2.318   
NORTH_MACEDONIA                    0.222           0.272             0.222   
NORWAY                             0.584           2.636             0.630   
POLAND                            14.486          15.384            13.127   
PORTUGAL                           1.101           2.468             2.014   
ROMANIA                            2.477           2.747             3.259   
SERBIA                             0.724           0.776             0.776   
SLOVAKIA                           0.924           1.023             1.180   
SLOVENIA                           0.611           0.645             0.611   
SPAIN                              6.596          14.461             8.472   
SWEDEN                             6.555           8.070             4.561   
SWITZERLAND                        0.433           0.665             0.512   
UNITED_KINGDOM                     6.070          10.695             6.070   
Total                            102.665         156.394            95.050   

                                        
                       Total installed  
Country                                 
ALBANIA                          0.214  
AUSTRIA                          1.743  
BELGIUM                          1.541  
BOSNIA_AND_HERZEGOVINA           0.109  
BULGARIA                         1.762  
CROATIA                          0.740  
CYPRUS                           0.402  
CZECH_REPUBLIC                   0.825  
DENMARK                          0.742  
ESTONIA                          0.177  
FINLAND                          6.780  
FRANCE                           9.878  
GERMANY                         22.818  
GREECE                           3.914  
HUNGARY                          1.421  
IRELAND                          1.577  
ITALY                           14.108  
LATVIA                           0.064  
LITHUANIA                        0.275  
LUXEMBOURG                       0.505  
MALTA                            0.085  
MONTENEGR

Heat pump: installed capacity per country (GW)


2030                              2035  \
                       New (accumulated) Total installed New (accumulated)   
Country                                                                      
ALBANIA                            0.812           0.814             0.846   
AUSTRIA                           20.681          20.682            20.916   
BELGIUM                           22.001          22.001            23.053   
BOSNIA_AND_HERZEGOVINA             1.881           1.881             1.950   
BULGARIA                           6.025           6.026             6.413   
CROATIA                            5.398           5.411             5.589   
CYPRUS                             0.534           0.536             0.590   
CZECH_REPUBLIC                    17.830          17.832            18.543   
DENMARK                            9.794           9.834            10.390   
ESTONIA                            2.151           2.151             2.191   
FINLAND                           15.894          18.022            17.262   
FRANCE                            95.328          95.537            97.420   
GERMANY                          149.400         150.118           160.623   
GREECE                             8.907           8.941             9.738   
HUNGARY                           18.822          18.823            19.373   
IRELAND                            7.796           7.808             8.558   
ITALY                             83.273          83.457            87.490   
LATVIA                             2.693           2.693             2.749   
LITHUANIA                          3.603           3.603             3.648   
LUXEMBOURG                         1.842           1.842             1.935   
MALTA                              0.210           0.212             0.235   
MONTENEGRO                         0.244           0.244             0.258   
NETHERLANDS                       25.414          25.414            26.282   
NORTH_MACEDONIA                    0.603           0.604             0.631   
NORWAY                             5.854           6.296             6.330   
POLAND                            38.920          38.924            41.228   
PORTUGAL                           5.129           5.161             5.408   
ROMANIA                           18.988          18.990            19.819   
SERBIA                             3.996           3.997             4.193   
SLOVAKIA                           7.695           7.696             8.152   
SLOVENIA                           3.039           3.039             3.138   
SPAIN                             31.501          31.648            35.469   
SWEDEN                            15.133          16.881            16.353   
SWITZERLAND                       16.197          16.197            16.437   
UNITED_KINGDOM                    93.242          93.412            96.699   
Total                            740.833         746.727           779.906   

                                        
                       Total installed  
Country                                 
ALBANIA                          0.847  
AUSTRIA                         20.917  
BELGIUM                         23.053  
BOSNIA_AND_HERZEGOVINA           1.950  
BULGARIA                         6.414  
CROATIA                          5.597  
CYPRUS                           0.591  
CZECH_REPUBLIC                  18.543  
DENMARK                         10.419  
ESTONIA                          2.191  
FINLAND                         18.441  
FRANCE                          97.534  
GERMANY                        161.014  
GREECE                           9.756  
HUNGARY                         19.373  
IRELAND                          8.564  
ITALY                           87.591  
LATVIA                           2.749  
LITHUANIA                        3.648  
LUXEMBOURG                       1.935  
MALTA                            0.236  
MONTENEGR

Electrolyzer: installed capacity per country (GW)


2030                              2035  \
               New (accumulated) Total installed New (accumulated)   
Country                                                              
AUSTRIA                    0.000           1.000             0.000   
BELGIUM                    0.000           0.000             0.000   
DENMARK                    0.000           5.652             0.000   
ESTONIA                    0.002           0.002             0.002   
FINLAND                    0.353           0.353             0.353   
FRANCE                     0.000           6.500             0.000   
GERMANY                    0.056           6.554             0.056   
ITALY                      0.000           5.000             0.000   
LATVIA                     0.002           0.002             0.002   
LITHUANIA                  0.198           0.198             0.198   
NETHERLANDS                0.000           3.500             0.000   
NORWAY                     0.194           0.194             0.194   
POLAND                     1.078           3.078             1.718   
PORTUGAL                   0.000           2.300             0.000   
SPAIN                      0.000           4.000             0.000   
SWEDEN                     0.000           1.860             0.000   
SWITZERLAND                0.001           0.001             0.001   
UNITED_KINGDOM             0.034           0.034             0.034   
Total                      1.919          40.229             2.559   

                                
               Total installed  
Country                         
AUSTRIA                  1.000  
BELGIUM                  0.000  
DENMARK                  6.499  
ESTONIA                  0.002  
FINLAND                  0.353  
FRANCE                   6.500  
GERMANY                  6.554  
ITALY                    5.000  
LATVIA                   0.002  
LITHUANIA                0.198  
NETHERLANDS              3.500  
NORWAY                   0.194  
POLAND                   3.718  
PORTUGAL                 2.300  
SPAIN                    4.000  
SWEDEN                   1.961  
SWITZERLAND              0.001  
UNITED_KINGDOM           0.034  
Total                   41.817

## 2. Export TECHGROUPKPOT.inc for the sensitivity scenario
Caps the capacity of electrolyzers (`ELECTROLYZER`) and heat pumps (`HEATPUMP`) per region. Electric boilers are not included: they share the `BOILER` tech group with all fuel-fired boilers, so `TECHGROUPKPOT` cannot cap them separately.

`TECHGROUPKPOT` has no year dimension: the cap holds in every modelled year, and it limits exogenous capacity (minus decommissioning) plus the accumulated new investments. To restrict only 2030 and 2035, the cap per region is

`max over years (EXOGENOUS + DECOMMISSIONING) + ENDOGENOUS(2025) + SHARE x (max ENDOGENOUS(2030, 2035) - ENDOGENOUS(2025))`

so the investments of the first year (`FIRST_YEAR`, 2025) and the existing capacity always fit under the cap (no infeasibility, 2025 is not restricted), and only the **additional** new capacity in 2030 and 2035 is limited to `SHARE` (50 %) of what the unrestricted run added. A region with no capacity at all gets `eps` (= 0 MW).

`ELECTROLYZER` is an acronym (hydrogen add-on) but not a member of the `TECH_GROUP` set, so GAMS would reject it in `TECHGROUPKPOT`. The scenario therefore also gets copies of `TECH_GROUP.inc` and `TECH_GROUP_VALUE.inc` with `ELECTROLYZER` added. The `TECHGROUPKPOT.inc` is the base file (hydro reservoir limits kept) with this block added.

In [2]:
SHARE = 0.5                                              # new capacity allowed, as share of the new capacity in the unrestricted run
SENS_SCENARIOS = ["sc_base_dc", "sc_base_nodc", "sc_Flex_1_share100_cap125_12h"]      # scenario folders in Sensitivities; the model reads their data/ before base/data/
SENS_DIRS = [Path.cwd() / "Sensitivities" / name / "data" for name in SENS_SCENARIOS]
BASE_FILE = Path.cwd() / "base" / "data" / "TECHGROUPKPOT.inc"
TECH_GROUP_OF = {"Electrolyzer": "ELECTROLYZER", "Heat pump": "HEATPUMP"}

FIRST_YEAR = "2025"                                       # not restricted: its investments stay under the cap
CAP_YEARS = YEARS                                         # years in which new capacity is restricted
ENDOFMODEL = "all_endofmodel_fullweek.gdx"                # raw model gdx of the reference run (model units)

# The cap must be in the units of the model equation (QKTECHGROUPR): GKFX - VDECOM_EXO_ACCUM + VGKNACCUMNET. G_CAP_YCRAF is rescaled
# (electrolyzers: hydrogen output instead of electricity input, so it is a factor 1/efficiency lower), so the raw variables are read.
import gams.transfer as gt
container = gt.Container(str(Path.cwd() / RUN_FOLDER / "model" / ENDOFMODEL))
area_region = res.get_result("G_CAP_YCRAF").rename(columns=RENAME)[["Area", "Region"]].drop_duplicates().set_index("Area")["Region"]

def p2x_group(generation):
    """Capped TECH_GROUP of the generator. GNR_IMPORT_H2 is also in the group ELECTROLYZER in GDATA, but it is moved to BACKUP_HYDROGEN below, so it is not capped."""
    g = str(generation).upper()
    if g.startswith("GNR_ELYS"): return "ELECTROLYZER"
    if g.startswith("GNR_HP_"): return "HEATPUMP"
    return None

def model_var(name, column):
    t = container[name].records.rename(columns={container[name].records.columns[0]: "Year", container[name].records.columns[1]: "Area",
                                               container[name].records.columns[2]: "Generation"})
    t = t.assign(Year=t["Year"].astype(str), Region=t["Area"].map(area_region), Tech=t["Generation"].map(p2x_group))
    t = t.loc[t["Tech"].isin(["ELECTROLYZER", "HEATPUMP"]) & t["Year"].isin([FIRST_YEAR] + CAP_YEARS) & t["Region"].notna()]
    return t.groupby(["Region", "Tech", "Year"])[column].sum()

gkfx = model_var("GKFX", "value")                         # exogenous capacity (MW)
decom = model_var("VDECOM_EXO_ACCUM", "level")            # accumulated decommissioning of exogenous capacity (MW)
endo = model_var("VGKNACCUMNET", "level").unstack("Year", fill_value=0)      # accumulated new investments (MW)
existing = gkfx.sub(decom, fill_value=0).groupby(["Region", "Tech"]).max().clip(lower=0)       # over the years, incl. FIRST_YEAR
endo_first = endo.get(FIRST_YEAR, 0)                       # accumulated new capacity before the restricted years
added = (endo[[y for y in CAP_YEARS if y in endo]].max(axis=1) - endo_first).clip(lower=0)      # extra new capacity in the restricted years
limit_mw = (existing.add(endo_first, fill_value=0) + SHARE * added.reindex(existing.index, fill_value=0)).unstack("Tech")
limit_mw = limit_mw.reindex(columns=["ELECTROLYZER", "HEATPUMP"]).fillna(0)
# Safety margin: a cap exactly equal to the existing capacity is infeasible (presolve: "Row QKTECHGROUPR(...) infeasible") because of
# rounding and solver tolerances in GKFX - VDECOM_EXO_ACCUM. Regions without any capacity keep eps (= 0 MW).
MARGIN_MW = 1.0
limit_mw = (limit_mw + MARGIN_MW * (limit_mw > 0)).round(1)

# GAMS table block, aligned with the base file (first column starts at character 107, 24 characters per column)
LABEL_W, COL_W = 106, 24
fmt = lambda v: "eps" if v == 0 else f"{v:g}"
lines = ["+".ljust(LABEL_W) + "".join(t.ljust(COL_W) for t in limit_mw.columns).rstrip()]
for region, row in limit_mw.sort_index().iterrows():
    lines.append(str(region).ljust(LABEL_W) + "".join(fmt(row[t]).ljust(COL_W) for t in limit_mw.columns).rstrip())
block = [f"*Sensitivity: electrolyzers and heat pumps, new capacity in {', '.join(CAP_YEARS)} limited to {SHARE:.0%} of the additional capacity of {RUN_FOLDER} (MW per region)"] + lines

base_lines = BASE_FILE.read_text().split("\n")
insert_at = next(i for i, l in enumerate(base_lines) if l.startswith("*National potentials"))   # before the (inactive) national potentials
out = base_lines[:insert_at] + block + base_lines[insert_at:]

# ELECTROLYZER is an acronym but not yet a TECH_GROUP member: add it (and its value) in copies of the base files, keeping their line endings
group_files = {}
for name in ("TECH_GROUP.inc", "TECH_GROUP_VALUE.inc"):
    text = (Path.cwd() / "base" / "data" / name).read_bytes().decode()
    if "ELECTROLYZER" in text:
        continue
    eol = "\r\n" if "\r\n" in text else "\n"
    rows = text.split(eol)
    last = max(i for i, r in enumerate(rows) if r.strip() == "/;")                    # closing line of the set / parameter
    rows.insert(last, "ELECTROLYZER".ljust(32) + ("ELECTROLYZER" if "VALUE" in name else ""))
    group_files[name] = eol.join(rows).encode()

# The hydrogen import GNR_IMPORT_H2 is in the same tech group (ELECTROLYZER) as the electrolyzers in HYDROGEN_GDATA.inc. To cap only the
# electrolyzers, its GDTECHGROUP is changed to BACKUP_HYDROGEN in a copy of the file. That group has no cap, and OUTPUT_SUMMARY.inc maps it to the
# same result technology type ELECTROLYZER, so the results (G_CAP, PRO, F_CONS, ECO_G) still contain the import. DUMMY would drop it from the results.
gdata = (Path.cwd() / "base" / "data" / "HYDROGEN_GDATA.inc").read_bytes().decode("latin-1")
eol = "\r\n" if "\r\n" in gdata else "\n"
gdata_rows = gdata.split(eol)
tg_col = next(l.index("GDTECHGROUP") for l in gdata_rows if "GDTECHGROUP" in l and not l.startswith("*"))
for i, l in enumerate(gdata_rows):
    if l.startswith("GNR_IMPORT_H2") and l[tg_col:tg_col + 12] == "ELECTROLYZER":
        gdata_rows[i] = l[:tg_col] + "BACKUP_HYDROGEN" + l[tg_col + 15:]
group_files["HYDROGEN_GDATA.inc"] = eol.join(gdata_rows).encode("latin-1")

for sens_dir in SENS_DIRS:
    sens_dir.mkdir(parents=True, exist_ok=True)
    (sens_dir / "TECHGROUPKPOT.inc").write_text("\n".join(out))
    print(f"Written {sens_dir / 'TECHGROUPKPOT.inc'}")
    for name, content in group_files.items():
        (sens_dir / name).write_bytes(content)
        print(f"Written {sens_dir / name}")
display(limit_mw)


Written /work3/s243704/Balmorel/Sensitivities/sc_restrict_p2x/data/TECHGROUPKPOT.inc
Written /work3/s243704/Balmorel/Sensitivities/sc_restrict_p2x/data/TECH_GROUP.inc
Written /work3/s243704/Balmorel/Sensitivities/sc_restrict_p2x/data/TECH_GROUP_VALUE.inc


Tech,ELECTROLYZER,HEATPUMP
Region,,
AL,0.0,798.6
AT,1000.0,20271.9
BA,0.0,1849.0
BE,0.2,21368.2
BG,0.0,5947.2
CH,1.4,15919.0
CY,0.0,522.5
CZ,0.0,17617.4
DE4-E,1992.0,24155.7
